# 📄 arXiv 論文爬蟲 — 物件導向版本
### 領域：電腦視覺 / 圖形辨識（Computer Vision & Image Recognition）

---

## 📌 這份 Notebook 的學習路徑

```
階段一：認識 arXiv API 網址長什麼樣
    ↓
階段二：用 Python 發送請求，看看伺服器回傳什麼
    ↓
階段三：認識 BeautifulSoup，了解它怎麼解析 XML
    ↓
階段四：用 BeautifulSoup 逐步取出每個欄位
    ↓
階段四點五：先手動走一次完整流程，存成 CSV
    ↓
階段四點七五：用 PyMySQL 手動存進資料庫
    ↓
階段五：用物件導向（Class）整理程式碼
    ↓
階段六：執行爬蟲，搜尋 CV 論文，儲存結果
```

---

## 📌 函式版本 vs 物件導向版本的差別

| | 函式版本 | 物件導向版本 |
|---|---|---|
| 組織方式 | 一個獨立函式 | 把資料和方法包在 Class 裡 |
| 狀態保存 | 每次呼叫都要重新傳參數 | 資料存在物件裡，隨時可以取用 |
| 擴充性 | 需要新增更多函式 | 直接在 Class 裡新增方法 |
| 適合場景 | 簡單單次任務 | 需要管理狀態、多次操作 |

---
# 階段一：認識 arXiv API 網址

在寫任何程式之前，先用**瀏覽器**直接打開下面這個網址，看看 API 回傳什麼：

👉 http://export.arxiv.org/api/query?search_query=cat:cs.CV&max_results=2

---

### 網址的組成結構

```
http://export.arxiv.org/api/query
        ↑ 這是 API 的基本網址（Base URL）

?search_query=cat:cs.CV&max_results=2
 ↑ 問號後面是參數，用 & 分隔多個參數
```

| 參數 | 意思 | 範例值 |
|------|------|--------|
| `search_query` | 搜尋什麼 | `cat:cs.CV`（電腦視覺分類） |
| `max_results` | 要幾筆資料 | `10` |
| `start` | 從第幾筆開始 | `0`（第一頁） |
| `sortBy` | 排序方式 | `submittedDate` |
| `sortOrder` | 升冪或降冪 | `descending`（最新優先） |

---
# 階段二：用 Python 發送請求

In [ ]:
!pip install requests pandas beautifulsoup4 lxml

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import pymysql
import configparser   # 讀取設定檔用

print('✅ 所有套件載入成功')

In [ ]:
# ▶ 發送一次請求，只要 2 筆資料

url = 'http://export.arxiv.org/api/query?search_query=cat:cs.CV&max_results=2'
response = requests.get(url)

print('HTTP 狀態碼：', response.status_code)
# 200 = 成功，429 = 請求太頻繁，500 = 伺服器錯誤

In [ ]:
# ▶ 看看回傳的原始內容（前 300 字元）

print(response.text[:300])

In [ ]:
# ▶ 印出完整 XML

print(response.text)

---
# 階段三：認識 BeautifulSoup

In [ ]:
# ▶ 用 BeautifulSoup 解析 response.text

soup    = BeautifulSoup(response.text, 'xml')
entries = soup.find_all('entry')

print('找到幾個 entry：', len(entries))

---
# 階段四：逐步取出每個欄位

In [ ]:
# ▶ 取出第一篇論文的各個欄位

entry_1 = entries[0]

title     = entry_1.find('title').text.strip()
summary   = entry_1.find('summary').text.strip()
published = entry_1.find('published').text[:10]
link      = entry_1.find('id').text

authors_list = []
for a in entry_1.find_all('author'):
    authors_list.append(a.find('name').text)

print('📌 標題  ：', title)
print('📅 日期  ：', published)
print('👤 作者  ：', authors_list)
print('🔗 連結  ：', link)
print('\n📝 摘要：')
print(summary)

---
# 階段四點五：先把探索階段的資料存成 CSV

在進入物件導向之前，先用我們剛才取出的資料，
手動走一次「收集所有論文 → 存成 CSV」的完整流程。

這樣做的目的是：
- 確認資料取出來是正確的
- 理解整個流程的每一步
- 之後把這些步驟搬進 Class 裡的時候，就知道在做什麼

In [ ]:
# ▶ 步驟一：用 for 迴圈收集所有論文資料
# entry 每一輪都換成不同的論文，所以所有取值都要在迴圈裡面

papers = []

for entry in entries:

    # 作者需要在迴圈裡面重新建立，每篇論文都不同
    authors_list = []
    for a in entry.find_all('author'):
        authors_list.append(a.find('name').text)

    paper = {
        '論文名'  : entry.find('title').text.strip(),
        '摘要'    : entry.find('summary').text.strip(),
        '發表時間' : entry.find('published').text[:10],
        '作者'    : authors_list,
        '論文網址' : entry.find('id').text
    }
    papers.append(paper)

print(f'✅ 共收集到 {len(papers)} 篇論文')

In [ ]:
# ▶ 步驟二：轉成 DataFrame，確認資料正確

df = pd.DataFrame(papers)
df

In [ ]:
# ▶ 步驟三：確認每篇論文的標題和作者

for i, paper in enumerate(papers):
    print(f'第 {i+1} 篇')
    print(f'  標題：{paper["論文名"]}')
    print(f'  作者：{paper["作者"]}')
    print()

In [ ]:
# ▶ 步驟四：儲存成 CSV
# utf-8-sig：讓 Excel 開啟時中文不會亂碼

df.to_csv('cv_papers_stage4.csv', index=False, encoding='utf-8-sig')
print(f'✅ 已儲存！共 {len(papers)} 筆論文 → cv_papers_stage4.csv')

---
## 📌 流程回顧

剛才我們手動完成了：

```
發送請求（requests.get）
    ↓
解析 XML（BeautifulSoup）
    ↓
找到所有 entry（soup.find_all）
    ↓
逐篇取出欄位（for entry in entries）
    ↓
收集成串列（papers.append）
    ↓
轉成 DataFrame（pd.DataFrame）
    ↓
儲存 CSV（df.to_csv）
```

接下來先把資料存進 MySQL，再進入階段五。

---
# 階段四點七五：用 PyMySQL 手動存進資料庫

在進入物件導向之前，先手動走一次「連線資料庫 → 建立資料表 → 存入資料 → 查詢」的完整流程。

這樣做的目的是：
- 確認 MySQL 連線正常
- 理解每個 SQL 語法在做什麼
- 之後把這些步驟搬進 Class 時，就知道在做什麼

---

## 資料庫設計

```
資料庫：arxiv_db
│
├── 資料表 papers（論文主表）
│   ├── id          INT, 自動編號
│   ├── title       VARCHAR(500), 論文標題
│   ├── summary     TEXT, 摘要
│   ├── published   DATE, 發表日期
│   ├── link        VARCHAR(200), 論文網址（唯一值，防止重複）
│   └── created_at  TIMESTAMP, 存入時間
│
├── 資料表 authors（作者表）
│   ├── id          INT, 自動編號
│   ├── paper_id    INT, 對應 papers 的 id
│   └── name        VARCHAR(200), 作者姓名
│
└── 資料表 search_logs（搜尋紀錄表）
    ├── id           INT, 自動編號
    ├── query        VARCHAR(200), 搜尋關鍵字
    ├── result_count INT, 新增幾筆
    └── searched_at  TIMESTAMP, 搜尋時間
```

In [ ]:
# 安裝 PyMySQL
!pip install pymysql

In [ ]:
import pymysql
import configparser   # 讀取設定檔用

print('✅ PyMySQL 載入成功')

---
## 步驟一：建立 config.ini 設定檔並連線到 MySQL

In [ ]:
# ▶ 先在跟這個 Notebook 同一個資料夾裡，新增一個 config.ini 檔案
# 內容如下：
#
# [DB]
# host     = localhost
# user     = root
# password = 你的密碼
# port     = 3306
# database = arxiv_db
#
# 好處：密碼不會直接寫在程式碼裡，上傳 GitHub 也不怕外洩

# ▶ 讀取設定檔
config = configparser.ConfigParser()
config.read('config.ini', encoding='utf-8')

print('✅ 設定檔讀取成功')
print('host    :', config.get('DB', 'host'))
print('user    :', config.get('DB', 'user'))
print('port    :', config.getint('DB', 'port'))
print('database:', config.get('DB', 'database'))
# 密碼不印出來，避免不小心外洩

In [ ]:
# ▶ 從設定檔讀取帳密，建立連線

conn = pymysql.connect(
    host        = config.get('DB', 'host'),
    user        = config.get('DB', 'user'),
    password    = config.get('DB', 'password'),
    port        = config.getint('DB', 'port'),
    charset     = 'utf8mb4',
    cursorclass = pymysql.cursors.DictCursor   # 回傳字典格式，用欄位名稱取值
)

print('✅ MySQL 連線成功')

---
## 步驟二：建立資料庫和資料表

In [ ]:
# ▶ 建立資料庫
# IF NOT EXISTS：如果已經存在就不重複建立，不會報錯

with conn.cursor() as cursor:        # with 語法：自動管理 cursor 的開啟和關閉
    sql = 'CREATE DATABASE IF NOT EXISTS arxiv_db'
    cursor.execute(sql)

    sql = 'USE arxiv_db'
    cursor.execute(sql)
# with 區塊結束，cursor 自動關閉

print('✅ 資料庫 arxiv_db 建立完成')

In [ ]:
# ▶ 建立論文主表 papers

with conn.cursor() as cursor:        # with 語法：自動管理 cursor 的開啟和關閉
    sql = '''
        CREATE TABLE IF NOT EXISTS papers (
            id         INT AUTO_INCREMENT PRIMARY KEY,
            title      VARCHAR(500),
            summary    TEXT,
            published  DATE,
            link       VARCHAR(200) UNIQUE,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    '''
    # UNIQUE：link 欄位不能重複，自動防止同一篇論文存入兩次
    cursor.execute(sql)
# with 區塊結束，cursor 自動關閉

print('✅ 資料表 papers 建立完成')

In [ ]:
# ▶ 建立作者表 authors
# 一篇論文可能有多位作者，所以獨立成一張表

with conn.cursor() as cursor:        # with 語法：自動管理 cursor 的開啟和關閉
    sql = '''
        CREATE TABLE IF NOT EXISTS authors (
            id       INT AUTO_INCREMENT PRIMARY KEY,
            paper_id INT,
            name     VARCHAR(200),
            FOREIGN KEY (paper_id) REFERENCES papers(id)
        )
    '''
    # FOREIGN KEY：paper_id 對應到 papers 資料表的 id
    # 確保作者一定對應到存在的論文
    cursor.execute(sql)
# with 區塊結束，cursor 自動關閉

print('✅ 資料表 authors 建立完成')

In [ ]:
# ▶ 建立搜尋紀錄表 search_logs

with conn.cursor() as cursor:        # with 語法：自動管理 cursor 的開啟和關閉
    sql = '''
        CREATE TABLE IF NOT EXISTS search_logs (
            id           INT AUTO_INCREMENT PRIMARY KEY,
            query        VARCHAR(200),
            result_count INT,
            searched_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    '''
    cursor.execute(sql)
# with 區塊結束，cursor 自動關閉

print('✅ 資料表 search_logs 建立完成')

---
## 步驟三：把論文資料存進資料庫

In [ ]:
# ▶ 逐篇把論文存進 papers 和 authors 資料表

new_count = 0   # 新增幾筆
dup_count = 0   # 重複幾筆（跳過）

with conn.cursor() as cursor:        # with 語法：自動管理 cursor 的開啟和關閉

    for paper in papers:
        try:
            # 新增論文主表
            sql = '''
                INSERT INTO papers (title, summary, published, link)
                VALUES (%s, %s, %s, %s)
            '''
            # %s 是佔位符號，防止 SQL injection
            # 實際的值由後面的 tuple 提供
            cursor.execute(sql, (
                paper['論文名'],
                paper['摘要'],
                paper['發表時間'],
                paper['論文網址']
            ))

            # 取得剛新增這筆論文的 id
            paper_id = cursor.lastrowid
            # lastrowid：上一個 INSERT 產生的自動編號

            # 新增作者表（每位作者一筆）
            sql = '''
                INSERT INTO authors (paper_id, name)
                VALUES (%s, %s)
            '''
            for author in paper['作者']:
                cursor.execute(sql, (paper_id, author))

            new_count += 1

        except pymysql.err.IntegrityError:
            # link 欄位重複（UNIQUE 限制），跳過這篇
            dup_count += 1

    # 記錄這次搜尋
    sql = '''
        INSERT INTO search_logs (query, result_count)
        VALUES (%s, %s)
    '''
    cursor.execute(sql, ('cat:cs.CV', new_count))

# with 區塊結束，cursor 自動關閉
# 確認寫入（沒有 commit 資料不會真的存進去）
conn.commit()

print(f'✅ 儲存完成！新增 {new_count} 筆，略過重複 {dup_count} 筆')

---
## 步驟四：從資料庫查詢資料

In [ ]:
# ▶ 查詢所有論文

with conn.cursor() as cursor:        # with 語法：自動管理 cursor 的開啟和關閉
    sql = '''
        SELECT id, title, published
        FROM papers
        ORDER BY published DESC
    '''
    cursor.execute(sql)
    results = cursor.fetchall()      # 取出所有結果（要在 with 裡面取）
# with 區塊結束，cursor 自動關閉

# DictCursor 回傳字典，直接用欄位名稱取值
df_db = pd.DataFrame(results)   # 不需要指定 columns，字典的 key 自動變欄位名稱
print(f'資料庫共有 {len(df_db)} 篇論文')
df_db

In [ ]:
# ▶ 查詢包含特定關鍵字的論文
# LIKE %關鍵字%：標題或摘要裡包含這個關鍵字

keyword = 'recognition'

with conn.cursor() as cursor:        # with 語法：自動管理 cursor 的開啟和關閉
    sql = '''
        SELECT title, published, link
        FROM papers
        WHERE title LIKE %s OR summary LIKE %s
        ORDER BY published DESC
    '''
    cursor.execute(sql, (f'%{keyword}%', f'%{keyword}%'))
    results = cursor.fetchall()      # 取出所有結果（要在 with 裡面取）
# with 區塊結束，cursor 自動關閉

# DictCursor 回傳字典，key 就是欄位名稱（title, published, link）
df_keyword = pd.DataFrame(results)
print(f'找到 {len(df_keyword)} 篇包含「{keyword}」的論文')
df_keyword

In [ ]:
# ▶ 查詢某篇論文的所有作者
# JOIN：把 papers 和 authors 兩張表合併查詢

with conn.cursor() as cursor:        # with 語法：自動管理 cursor 的開啟和關閉
    sql = '''
        SELECT p.title, a.name
        FROM papers p
        JOIN authors a ON p.id = a.paper_id
        ORDER BY p.id
    '''
    cursor.execute(sql)
    results = cursor.fetchall()      # 取出所有結果（要在 with 裡面取）
# with 區塊結束，cursor 自動關閉

# DictCursor 回傳字典，key 就是欄位名稱（title, name）
df_authors = pd.DataFrame(results)
df_authors

In [ ]:
# ▶ 查詢搜尋紀錄

with conn.cursor() as cursor:        # with 語法：自動管理 cursor 的開啟和關閉
    sql = '''
        SELECT query, result_count, searched_at
        FROM search_logs
        ORDER BY searched_at DESC
    '''
    cursor.execute(sql)
    results = cursor.fetchall()      # 取出所有結果（要在 with 裡面取）
# with 區塊結束，cursor 自動關閉

# DictCursor 回傳字典，key 就是欄位名稱（query, result_count, searched_at）
df_logs = pd.DataFrame(results)
df_logs

In [ ]:
# ▶ 關閉連線
# cursor 已經由 with 自動關閉，這裡只需要關閉連線
conn.close()
print('✅ 資料庫連線已關閉')

---
## 📌 MySQL 流程回顧

```
pymysql.connect()          → 連線 MySQL
    ↓
cursor.execute(CREATE)     → 建立資料庫和資料表
    ↓
cursor.execute(INSERT)     → 存入論文資料
    ↓
conn.commit()              → 確認寫入
    ↓
cursor.execute(SELECT)     → 查詢資料
    ↓
cursor.fetchall()          → 取出查詢結果
    ↓
conn.close()               → 關閉連線
```

接下來階段五就是把 CSV 和 MySQL 這兩個流程，**一起搬進 Class 裡面整理好**。

---
# 階段五：用物件導向（Class）整理程式碼

## 為什麼用物件導向？

函式版本每次呼叫都是獨立的，資料用完就不見了：

```python
# 函式版本
papers = fetch_arxiv_papers(query, max_results)
# 只能拿到 papers，其他資料（網址、參數）都不見了
```

物件導向版本把**資料和操作**都包在一起，可以隨時取用：

```python
# 物件導向版本
crawler = ArxivCrawler(max_results=10)
crawler.search('image recognition')   # 搜尋
crawler.papers                        # 取得論文資料
crawler.save_csv()                    # 儲存
crawler.search('object detection')    # 再搜尋新的
```

---

## Class 的基本結構

```python
class 類別名稱:

    def __init__(self, 初始參數):
        # 物件建立時執行，用來設定初始資料
        self.資料 = 初始值

    def 方法名稱(self):
        # 物件可以執行的動作
        pass
```

- `__init__`：物件建立時自動執行，設定初始狀態
- `self`：代表「這個物件自己」，用來存取物件內部的資料
- 方法（method）：物件可以執行的動作，就像函式但屬於這個 Class

In [ ]:
# configparser 在階段四點七五已經 import 過了
# 這裡確認一下都有載入
import pymysql
import configparser

print('✅ 確認套件載入完成，可以開始定義 Class')

In [ ]:
class ArxivCrawler:
    """
    arXiv 論文爬蟲
    負責搜尋論文、解析資料、儲存結果
    """

    def __init__(self, max_results=10):
        """
        物件建立時執行
        設定初始資料：API 網址、最大筆數、空的論文串列

        參數：
            max_results: 每次搜尋最多回傳幾筆，預設 10
        """
        # self.xxx 代表這個物件自己擁有的資料
        self.base_url    = 'http://export.arxiv.org/api/query'  # API 網址
        self.max_results = max_results                           # 最大筆數
        self.papers      = []                                    # 存放論文的串列
        self.query       = ''                                    # 目前的搜尋關鍵字
        self.db          = None                                  # 資料庫連線（預設未連線）

        print(f'✅ ArxivCrawler 建立完成，每次最多爬取 {self.max_results} 筆')


    def search(self, query):
        """
        發送請求，搜尋論文

        參數：
            query: 搜尋關鍵字，例如 'cat:cs.CV AND image recognition'
        """
        self.query = query   # 把搜尋關鍵字存起來，之後可以查看

        # 設定請求參數
        params = {
            'search_query': query,
            'start'       : 0,
            'max_results' : self.max_results,
            'sortBy'      : 'submittedDate',
            'sortOrder'   : 'descending'
        }

        # 發送請求
        print(f'🔍 搜尋中：{query}')
        response = requests.get(self.base_url, params=params)

        # 確認請求成功
        if response.status_code != 200:
            print(f'❌ 請求失敗，狀態碼：{response.status_code}')
            return

        # 解析 XML
        self.papers = self._parse(response.text)
        print(f'✅ 搜尋完成，共取得 {len(self.papers)} 篇論文')


    def _parse(self, xml_text):
        """
        解析 XML，取出每篇論文的資料
        方法名稱前面加底線 _ 代表這是內部使用的方法，不需要從外部呼叫

        參數：
            xml_text: 從 API 回傳的 XML 字串（response.text）
        回傳：
            papers: 論文資料的串列
        """
        soup    = BeautifulSoup(xml_text, 'xml')
        entries = soup.find_all('entry')

        papers = []
        for entry in entries:

            # 取出作者串列
            authors_list = []
            for a in entry.find_all('author'):
                authors_list.append(a.find('name').text)

            paper = {
                '論文名'  : entry.find('title').text.strip(),
                '摘要'    : entry.find('summary').text.strip(),
                '發表時間' : entry.find('published').text[:10],
                '作者'    : authors_list,
                '論文網址' : entry.find('id').text
            }
            papers.append(paper)

        return papers


    def show(self):
        """
        把論文資料顯示成 DataFrame 表格
        """
        if not self.papers:
            print('⚠️ 還沒有資料，請先執行 search()')
            return

        df = pd.DataFrame(self.papers)
        return df


    def show_paper(self, index=0):
        """
        顯示單篇論文的詳細資訊

        參數：
            index: 第幾篇論文，預設第一篇（index=0）
        """
        if not self.papers:
            print('⚠️ 還沒有資料，請先執行 search()')
            return

        paper = self.papers[index]
        print(f'📌 標題  ：{paper["論文名"]}')
        print(f'📅 日期  ：{paper["發表時間"]}')
        print(f'👤 作者  ：{", ".join(paper["作者"])}')
        print(f'🔗 連結  ：{paper["論文網址"]}')
        print(f'\n📝 摘要：')
        print(paper['摘要'])


    def save_csv(self, filename='cv_papers.csv'):
        """
        把論文資料儲存成 CSV 檔案

        參數：
            filename: 檔案名稱，預設 'cv_papers.csv'
        """
        if not self.papers:
            print('⚠️ 還沒有資料，請先執行 search()')
            return

        df = pd.DataFrame(self.papers)
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f'✅ 已儲存！共 {len(self.papers)} 筆論文 → {filename}')


    def connect_db(self, config_file='config.ini', database='arxiv_db'):
        """
        從設定檔讀取帳密，連線到 MySQL 資料庫
        連線成功後自動建立資料庫和資料表（如果還不存在）

        參數：
            config_file: 設定檔路徑，預設 'config.ini'
            database   : 資料庫名稱，預設 'arxiv_db'
        """
        # 讀取設定檔
        config = configparser.ConfigParser()
        config.read(config_file, encoding='utf-8')

        # 從設定檔取得帳密，建立連線
        self.db = pymysql.connect(
            host        = config.get('DB', 'host'),
            user        = config.get('DB', 'user'),
            password    = config.get('DB', 'password'),
            port        = config.getint('DB', 'port'),
            charset     = 'utf8mb4',
            cursorclass = pymysql.cursors.DictCursor   # 回傳字典格式
        )

        # with 語法：自動管理 cursor 的開啟和關閉
        with self.db.cursor() as cursor:

            # 建立資料庫（如果不存在）
            sql = f'CREATE DATABASE IF NOT EXISTS {database}'
            cursor.execute(sql)

            sql = f'USE {database}'
            cursor.execute(sql)

            # 建立論文主表
            sql = '''
                CREATE TABLE IF NOT EXISTS papers (
                    id         INT AUTO_INCREMENT PRIMARY KEY,
                    title      VARCHAR(500),
                    summary    TEXT,
                    published  DATE,
                    link       VARCHAR(200) UNIQUE,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            '''
            cursor.execute(sql)

            # 建立作者表
            sql = '''
                CREATE TABLE IF NOT EXISTS authors (
                    id       INT AUTO_INCREMENT PRIMARY KEY,
                    paper_id INT,
                    name     VARCHAR(200),
                    FOREIGN KEY (paper_id) REFERENCES papers(id)
                )
            '''
            cursor.execute(sql)

            # 建立搜尋紀錄表
            sql = '''
                CREATE TABLE IF NOT EXISTS search_logs (
                    id           INT AUTO_INCREMENT PRIMARY KEY,
                    query        VARCHAR(200),
                    result_count INT,
                    searched_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            '''
            cursor.execute(sql)

        # with 區塊結束，cursor 自動關閉
        self.db.commit()
        print(f'✅ 資料庫連線成功，資料表已準備完成')


    def save_db(self):
        """
        把目前的搜尋結果存進 MySQL
        重複的論文（依 link 判斷）自動跳過
        """
        if not self.papers:
            print('⚠️ 還沒有資料，請先執行 search()')
            return

        if not self.db:
            print('⚠️ 尚未連線資料庫，請先執行 connect_db()')
            return

        new_count = 0   # 新增幾筆
        dup_count = 0   # 重複幾筆

        # with 語法：自動管理 cursor 的開啟和關閉
        with self.db.cursor() as cursor:

            for paper in self.papers:
                try:
                    # 新增論文主表
                    sql = '''
                        INSERT INTO papers (title, summary, published, link)
                        VALUES (%s, %s, %s, %s)
                    '''
                    cursor.execute(sql, (
                        paper['論文名'],
                        paper['摘要'],
                        paper['發表時間'],
                        paper['論文網址']
                    ))

                    # 取得剛新增這筆論文的 id
                    paper_id = cursor.lastrowid

                    # 新增作者表（每位作者一筆）
                    sql = '''
                        INSERT INTO authors (paper_id, name)
                        VALUES (%s, %s)
                    '''
                    for author in paper['作者']:
                        cursor.execute(sql, (paper_id, author))

                    new_count += 1

                except pymysql.err.IntegrityError:
                    # link 重複，跳過這篇
                    dup_count += 1

            # 記錄這次搜尋
            sql = '''
                INSERT INTO search_logs (query, result_count)
                VALUES (%s, %s)
            '''
            cursor.execute(sql, (self.query, new_count))

        # with 區塊結束，cursor 自動關閉
        self.db.commit()
        print(f'✅ 儲存完成！新增 {new_count} 筆，略過重複 {dup_count} 筆')


    def show_all_papers(self):
        """
        從資料庫查詢所有論文
        對應階段四點七五的「查詢所有論文」
        """
        if not self.db:
            print('⚠️ 尚未連線資料庫，請先執行 connect_db()')
            return

        # with 語法：自動管理 cursor 的開啟和關閉
        with self.db.cursor() as cursor:
            sql = '''
                SELECT id, title, published
                FROM papers
                ORDER BY published DESC
            '''
            cursor.execute(sql)
            results = cursor.fetchall()

        # with 區塊結束，cursor 自動關閉
        # DictCursor 回傳字典，key 就是欄位名稱（id, title, published）
        df = pd.DataFrame(results)
        print(f'✅ 資料庫共有 {len(df)} 篇論文')
        return df


    def query_db(self, keyword):
        """
        從資料庫查詢包含關鍵字的論文

        參數：
            keyword: 要查詢的關鍵字
        """
        if not self.db:
            print('⚠️ 尚未連線資料庫，請先執行 connect_db()')
            return

        # with 語法：自動管理 cursor 的開啟和關閉
        with self.db.cursor() as cursor:
            sql = '''
                SELECT title, published, link
                FROM papers
                WHERE title LIKE %s OR summary LIKE %s
                ORDER BY published DESC
            '''
            cursor.execute(sql, (f'%{keyword}%', f'%{keyword}%'))
            results = cursor.fetchall()

        # with 區塊結束，cursor 自動關閉
        # DictCursor 回傳字典，key 就是欄位名稱（title, published, link）
        df = pd.DataFrame(results)
        print(f'✅ 找到 {len(df)} 篇包含「{keyword}」的論文')
        return df


    def show_authors(self, title_keyword):
        """
        查詢包含特定關鍵字的論文的所有作者
        對應階段四點七五的「查詢某篇論文的所有作者」
        使用 JOIN 把 papers 和 authors 兩張表合併查詢

        參數：
            title_keyword: 論文標題的關鍵字
        """
        if not self.db:
            print('⚠️ 尚未連線資料庫，請先執行 connect_db()')
            return

        # with 語法：自動管理 cursor 的開啟和關閉
        with self.db.cursor() as cursor:
            sql = '''
                SELECT p.title, a.name
                FROM papers p
                JOIN authors a ON p.id = a.paper_id
                WHERE p.title LIKE %s
                ORDER BY p.id
            '''
            # JOIN：把 papers（p）和 authors（a）兩張表合併
            # ON p.id = a.paper_id：兩張表的對應關係
            cursor.execute(sql, (f'%{title_keyword}%',))
            results = cursor.fetchall()

        # with 區塊結束，cursor 自動關閉
        # DictCursor 回傳字典，key 就是欄位名稱（title, name）
        df = pd.DataFrame(results)
        print(f'✅ 找到 {len(df)} 筆包含「{title_keyword}」的論文作者資料')
        return df


    def show_logs(self):
        """
        查詢並顯示所有搜尋紀錄
        """
        if not self.db:
            print('⚠️ 尚未連線資料庫，請先執行 connect_db()')
            return

        # with 語法：自動管理 cursor 的開啟和關閉
        with self.db.cursor() as cursor:
            sql = '''
                SELECT query, result_count, searched_at
                FROM search_logs
                ORDER BY searched_at DESC
            '''
            cursor.execute(sql)
            results = cursor.fetchall()

        # with 區塊結束，cursor 自動關閉
        # DictCursor 回傳字典，key 就是欄位名稱（query, result_count, searched_at）
        df = pd.DataFrame(results)
        print(f'✅ 共有 {len(df)} 筆搜尋紀錄')
        return df


    def close_db(self):
        """
        關閉資料庫連線
        """
        if self.db:
            self.db.close()
            self.db = None
            print('✅ 資料庫連線已關閉')


print('✅ ArxivCrawler Class 定義完成')

---
## Class 結構總覽

```
ArxivCrawler
│
├── 資料（屬性）
│   ├── self.base_url     API 網址
│   ├── self.max_results  最大筆數
│   ├── self.papers       論文串列
│   ├── self.query        目前搜尋關鍵字
│   └── self.db           資料庫連線物件
│
└── 動作（方法）
    ├── 搜尋
    │   ├── __init__()    建立物件，設定初始資料
    │   ├── search()      發送請求，搜尋論文
    │   └── _parse()      解析 XML（內部使用）
    │
    ├── 顯示
    │   ├── show()        顯示成 DataFrame 表格
    │   └── show_paper()  顯示單篇論文詳細資訊
    │
    ├── 儲存
    │   └── save_csv()    儲存成 CSV 檔案
    │
    └── 資料庫
        ├── connect_db()       連線 MySQL，自動建立資料表
        ├── save_db()          存進資料庫，自動略過重複
        ├── show_all_papers()  查詢所有論文
        ├── query_db()         從資料庫查詢關鍵字
        ├── show_authors()     查詢論文的所有作者
        ├── show_logs()        查看搜尋紀錄
        └── close_db()         關閉連線
```

---
# 階段六：執行爬蟲，搜尋電腦視覺論文

In [ ]:
# ▶ 建立爬蟲物件
# 這裡會自動執行 __init__

crawler = ArxivCrawler(max_results=10)

In [ ]:
# ▶ 搜尋論文

crawler.search('cat:cs.CV AND (image recognition OR object detection)')

In [ ]:
# ▶ 顯示成表格

crawler.show()

In [ ]:
# ▶ 查看第一篇論文詳細資訊

crawler.show_paper(index=0)

In [ ]:
# ▶ 查看第二篇論文詳細資訊

crawler.show_paper(index=1)

In [ ]:
# ▶ 直接取得論文資料（self.papers 存在物件裡，隨時可以取用）

print('目前搜尋關鍵字：', crawler.query)
print('論文總篇數：', len(crawler.papers))
print('第一篇標題：', crawler.papers[0]['論文名'])

In [ ]:
# ▶ 換一個關鍵字搜尋（不需要重新建立物件）

time.sleep(3)  # 等 3 秒，避免請求太頻繁
crawler.search('cat:cs.CV AND image segmentation')

In [ ]:
# ▶ 儲存成 CSV

crawler.save_csv('cv_papers_oop.csv')

In [ ]:
# ▶ 多個關鍵字搜尋，合併結果

queries = [
    'cat:cs.CV AND image recognition',
    'cat:cs.CV AND object detection',
    'cat:cs.CV AND image segmentation'
]

all_papers = []

for q in queries:
    crawler.search(q)
    all_papers.extend(crawler.papers)   # 把每次結果合併
    time.sleep(3)

# 去除重複論文
df_all = pd.DataFrame(all_papers).drop_duplicates(subset='論文網址')
print(f'\n✅ 總共爬取到 {len(df_all)} 篇不重複論文')

# 儲存合併結果
df_all.to_csv('cv_papers_all.csv', index=False, encoding='utf-8-sig')
print('✅ 已儲存 → cv_papers_all.csv')

---
## MySQL 部分

In [ ]:
# ▶ 步驟一：連線資料庫
# 請確認同一個資料夾裡有 config.ini 設定檔

crawler.connect_db(
    config_file = 'config.ini',   # ← 設定檔路徑
    database    = 'arxiv_db'
)
# 連線成功後會自動建立 papers、authors、search_logs 三張資料表

In [ ]:
# ▶ 步驟二：搜尋論文並存進資料庫

crawler.search('cat:cs.CV AND image recognition')
crawler.save_db()   # 存進 MySQL，重複的自動跳過

In [ ]:
# ▶ 步驟三：換關鍵字再搜尋，繼續累積資料

time.sleep(3)
crawler.search('cat:cs.CV AND object detection')
crawler.save_db()   # 重複的論文自動跳過，不會存入兩次

In [ ]:
# ▶ 步驟四：從資料庫查詢包含關鍵字的論文

df_result = crawler.query_db('transformer')
df_result

In [ ]:
# ▶ 步驟四點一：查詢資料庫所有論文

df_all = crawler.show_all_papers()
df_all

In [ ]:
# ▶ 步驟四點二：查詢包含特定關鍵字的論文的所有作者
# 使用 JOIN 把 papers 和 authors 兩張表合併查詢

df_authors = crawler.show_authors('recognition')
df_authors

In [ ]:
# ▶ 步驟五：查看搜尋紀錄
# 每次 save_db() 都會自動記錄搜尋了什麼、新增了幾筆

crawler.show_logs()

In [ ]:
# ▶ 步驟六：關閉資料庫連線

crawler.close_db()